# PA4 Task 1 — mechanism diagnostic (supplementary, ~25–35 min on one T4)

Runs **after** the main Task 1 run and changes no condition. It loads the unchanged FP16 Qwen2.5-1.5B plus the AWQ
scales and calibration tokens the main run committed under `PA4/Task 1/results/`, and measures *why* some units are
fragile: the residual-stream map (massive activations, attention mass on token 0, which sublayer writes them),
the weight difficulty of every Q3 unit, one-unit-at-2-bits propagation traces, activation patching and weight
protection tests. Output: `results/diagnostics/mechanism_diagnostic.json`, then tables T20–T23 and figure F13.

**Before running:** the main run's `results/` folder and `atml-pa4-datasets/data/local_source_revisions.json` must be
committed and pushed (the notebook clones the repo). Settings: **Accelerator = GPU T4 (one is enough)**, **Internet = On**.

In [ ]:
REPO_URL = "https://github.com/wasaymj/ATML.git"
BRANCH   = "main"
N_SEQ    = 32          # calibration sequences used by the diagnostic (first 32 of the locked 128)

import os, subprocess, sys, shutil, json
WORK = "/kaggle/working"
REPO = f"{WORK}/ATML"
TASK = f"{REPO}/PA4/Task 1"
DATA = f"{REPO}/PA4/atml-pa4-datasets"
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
need = ["results/sensitivity/awq_scales.npz", "results/sensitivity/awq_search.json", "results/sensitivity/dense_input_norms.npz",
        "results/sensitivity/q3_sensitivity.json", "results/data_selection/c4_calibration_locked.jsonl.gz"]
missing = [p for p in need if not os.path.exists(f"{TASK}/{p}")] +           ([] if os.path.exists(f"{DATA}/data/local_source_revisions.json") else ["atml-pa4-datasets/data/local_source_revisions.json"])
assert not missing, f"commit and push these first: {missing}"

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{TASK}/requirements.txt"], check=True)
from huggingface_hub import snapshot_download
rev = json.load(open(f"{DATA}/data/local_source_revisions.json"))["revisions"]["model_base"]
print("Qwen/Qwen2.5-1.5B @", rev, "->", snapshot_download("Qwen/Qwen2.5-1.5B", revision=rev))

In [ ]:
env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", TOKENIZERS_PARALLELISM="false",
           HF_HUB_DISABLE_PROGRESS_BARS="1", CUDA_VISIBLE_DEVICES="0")
rc = subprocess.run([sys.executable, "scripts/diagnose_task1.py", "--n-seq", str(N_SEQ)], cwd=TASK, env=env).returncode
assert rc == 0, "diagnostic failed - read the log above"
subprocess.run([sys.executable, "scripts/compile_task1_report.py"], cwd=TASK, env=env)

In [ ]:
# package only the new outputs (plus regenerated tables/figures/analysis) for download
out = f"{WORK}/task1_diagnostics"
shutil.rmtree(out, ignore_errors=True)
for sub in ["diagnostics", "tables", "figures", "analysis"]:
    shutil.copytree(f"{TASK}/results/{sub}", f"{out}/results/{sub}")
shutil.make_archive(f"{WORK}/task1_diagnostics", "zip", out, "results")
print(sorted(os.listdir(WORK)))


In [ ]:
# inline preview (display only; the zip above is already written)
from IPython.display import Markdown, Image, display
for t in ["T20_massive_activations", "T21_propagation_and_patching", "T22_weight_protection"]:
    display(Markdown(f"### {t}" + chr(10) + open(f"{TASK}/results/tables/{t}.md", encoding="utf-8").read()))
display(Image(f"{TASK}/results/figures/F13_propagation_traces.png", width=1000))
